<a href="https://colab.research.google.com/github/Ashish2343/R-MoE-Architecture-Comparisions/blob/main/Data_preperation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import glob
import os

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
path = '/content/drive/MyDrive/AI Reasearch/CICIDS/MachineLearningCVE'
all_files = glob.glob(os.path.join(path, "*.csv"))

In [5]:
df_list = []
for filename in all_files:
    # Use low_memory=False to handle large files efficiently
    temp_df = pd.read_csv(filename, encoding='cp1252', low_memory=False)
    # CRITICAL: Strip spaces from column names (CICIDS has hidden spaces)
    temp_df.columns = temp_df.columns.str.strip()
    df_list.append(temp_df)

In [7]:
full_data = pd.concat(df_list, axis=0, ignore_index=True)
print(f"Total rows loaded: {len(full_data)}")

Total rows loaded: 2830743


In [8]:
# Handle Inf/NaN values globally
full_data.replace([np.inf, -np.inf], np.nan, inplace=True)
full_data.dropna(inplace=True)

In [9]:
# 3. LABEL CONSOLIDATION
full_data['Label'] = full_data['Label'].replace(['DoS Hulk', 'DoS GoldenEye', 'DoS slowloris', 'DoS Slowhttptest'], 'DoS')
full_data['Label'] = full_data['Label'].replace(['FTP-Patator', 'SSH-Patator', 'Web Attack  Brute Force'], 'BruteForce')

In [10]:
# Filter out very rare/noisy classes
unwanted = ['Web Attack  XSS', 'Web Attack  Sql Injection', 'Heartbleed', 'Infiltration']
full_data = full_data[~full_data['Label'].isin(unwanted)]

In [11]:
# 4. STRATEGIC RESAMPLING
def get_sample(df, label, n):
    subset = df[df['Label'] == label]
    return subset.sample(n=min(len(subset), n), random_state=42)

final_df = pd.concat([
    get_sample(full_data, 'BENIGN', 200_000),
    get_sample(full_data, 'DoS', 100_000),
    get_sample(full_data, 'PortScan', 60_000),
    get_sample(full_data, 'DDoS', 60_000),
    get_sample(full_data, 'BruteForce', 15_000),
    get_sample(full_data, 'Bot', 1_900),
]).reset_index(drop=True)

In [12]:
final_df.shape

(435732, 79)

In [ ]:
final_df.to_csv('/content/drive/MyDrive/AI Reasearch/CICIDS/Incremental01/final_data1.csv', index=False)